Notebook priprema i izvršava tri sistema:
1. **GTE Full** - proizvod predstavljen je embeddingom kompletnog dokumenta
2. **E5-MaxP** - proizvod je predstavljen skupom chunkova, a njegov konačni score se određuje kao MaxP (najveća sličnost chunka sa upitom)
3. **E5-MeanChunks** - embedding proizvoda je prosek embeddinga svih njegovih chunkova

Sva tri koriste multilingual bi encoder modele i FAISS IndexFlatIP indeks. Vektori se-normalizuju, a sličnost računa kao cosine, tj. skalarni proizvod prethodno normalizovanih vektora.

## 1. Priprema okruženja

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)
import torch
from huggingface_hub import model_info
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
import unicodedata
import re
import torch
from pathlib import Path
import sys
import faiss
import html

## 2. Konfiguracija 



In [26]:
if torch.cuda.is_available():
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path("/content/drive/MyDrive/RecSys")
else:
    PROJECT_ROOT = Path.cwd().parent
print('PROJECT_ROOT:', PROJECT_ROOT.resolve())
sys.path.insert(0, str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/RecSys


In [ ]:
SEED = 42
DEVICE = 'cuda'  if torch.cuda.is_available() else 'cpu'
GTE_MODEL_ID = 'Alibaba-NLP/gte-multilingual-base'
E5_MODEL_ID = 'intfloat/multilingual-e5-base'
GTE_BATCH_SIZE = 8 if DEVICE == 'cuda' else 2
E5_BATCH_SIZE = 16 if DEVICE == 'cuda' else 2
E5_INPUT_LIMIT=512
DATA_DIR = PROJECT_ROOT / 'data'
INDEX_DIR = DATA_DIR / 'indexes'
EVAL_DIR = DATA_DIR / 'evaluation'
INDEX_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PRODUCTS = 1_268
EMBEDDING_DIMENSION = 768
DEPTH = 5
E5_OVERLAP_TOKENS = 64


## 3. Učitavanje i obrada ulaznih podataka

Korišćeni ulazi:

1. product_documents.csv - dokumenti proizvoda
2. products_logical.csv - svi podaci o postojećim proizvodima 
3. researcher_test.csv - primeri korisničkih upita

In [28]:
def clean_query_text(value):
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
documents = pd.read_csv(DATA_DIR / 'processed' / 'product_documents.csv').fillna("")

products =pd.read_csv(DATA_DIR / 'processed' / 'products_logical.csv')
products["price"] = pd.to_numeric(products["price"], errors="raise")
products["page_rating"] = pd.to_numeric(products["page_rating"],errors="coerce")


queries = pd.read_csv(DATA_DIR / 'queries' / 'researcher_test.csv')
#upiti prolaze kroz istu obradu kroz koju su prošli proizvodi 
queries["query_text"] = queries["query_text"].map(clean_query_text)

display(queries['query_type'].value_counts().sort_index())
display(documents[['logical_product_id', 'document_text']].head(1))

,count
query_type,
broad_or_underspecified,9
multiple_requirements,13
problem_or_negative_constraint,14
single_property,12


,logical_product_id,document_text
0,P0001,"Name: 100% Natural All Aglow Lip & Cheek Stick\nBrand: Burt's Bees\nCategory: Blush\nDescription: Burt's Bees All Aglow Lip & Cheek Stick offers the convenience of an all in one hybrid lip and cheek color stick that delivers the perfect color pop. Details Benefits LIP & CHEEK TINT: This dual-function make up stick gives your lips and cheeks the perfect color pop in a naturally flattering Suez Sands shade that screams a soft pinkish brown tone NOURISHING INGREDIENTS: This makeup duo is formulated to hydrate skin with 100% natural ingredients such as Jojoba Seed Oil Sunflower Seed Oil and a hydrating core made of coconut oil EASY APPLICATION: Swipe the 2-in-1 stick along your cheekbones as a blush or dab on your lips for a healthy-looking lightweight lip color without a heavy or cakey feel COLOR THAT CARES: Applies smoothly and blends easily this lip and cheek color is available in a variety of shades ranging from nude pink red coral and plum that deliver natural-looking color to enhance your natural glow NATURAL MAKEUP: This cheek and lip stick is dermatologist-tested and naturally formulated without parabens, phthalates, SLS, petrolatum or synthetic fragrances. Never tested on animals How To Use Swipe lip and cheek stick along your cheekbones and dab on your lips for a healthy-looking glow. For external use only.\nPros: flattering/good shade match, blends easily, natural-looking finish, smooth/creamy texture, easy to apply/use, hydrating/non-drying, long-lasting wear\nCons: \nBest uses: cheeks/blush, lips/lip color, mature skin, travel/on-the-go touch-ups"


## 4. Provera kontekstnog prozora tokenizera

Broj tokena dokumenata se zasebno meri preko GTE i E5 tokenizerom, kako bi se utvrdilo da li se modelu može proslediti kompletan dokument ili je potrebno uvesti chunkovanje.

In [30]:
GTE_REVISION = model_info(GTE_MODEL_ID).sha
E5_REVISION = model_info(E5_MODEL_ID).sha

gte_tokenizer = AutoTokenizer.from_pretrained(GTE_MODEL_ID, revision=GTE_REVISION, trust_remote_code=True)
e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_ID, revision=E5_REVISION)

gte_lengths = np.array(gte_tokenizer(documents['document_text'].tolist(),return_length=True)["length"])
e5_full_lengths = np.array(e5_tokenizer(('passage: ' + documents['document_text']).tolist(), return_length=True)["length"])

print("GTE tokenizer limit exceeded" if gte_lengths.max() > gte_tokenizer.model_max_length else "GTE tokenizer limit satisfied")
print("E5 tokenizer limit exceeded" if e5_full_lengths.max() > e5_tokenizer.model_max_length else "E5 tokenizer limit satisfied")

print('GTE model:', GTE_MODEL_ID, 'GTE commit:', GTE_REVISION)
print('E5 model:', E5_MODEL_ID, 'E5 commit:', E5_REVISION)
print(f'E5 full documents over {e5_tokenizer.model_max_length}:',int(np.count_nonzero(e5_full_lengths > e5_tokenizer.model_max_length)))

Token indices sequence length is longer than the specified maximum sequence length for this model (679 > 512). Running this sequence through the model will result in indexing errors


GTE tokenizer limit satisfied
E5 tokenizer limit exceeded
GTE model: Alibaba-NLP/gte-multilingual-base GTE commit: 9bbca17d9273fd0d03d5725c7a4b0f6b45142062
E5 model: intfloat/multilingual-e5-base E5 commit: d128750597153bb5987e10b1c3493a34e5a4502a
E5 full documents over 512: 207


### Zaključak

GTE model može da predstavi svaki proizvod jednim embeddingom kompletnog dokumenta.

Kod E5 modela 207 od 1.268 dokumenata prelazi 512 tokena. Truncation bi bi moglo ukloniti opis ili platformske podakte relevantne za određeni upit.

Za E5 će biti korišćena hibridna strategija:

1. **structure-aware chunking** - opis i platformski podaci predstavljaju odvojene celine jednog prozivoda
2. **metadata enrichment** - naziv, brend i kategorija se unose u svaki chunku
3. **token-based sliding window** - predugačke celine se dele prema E5 tokenima, sa overlap-om od 64 tokena

## 5. GTE

### 5.1. GTE embedding proizvoda

In [ ]:
gte_model = SentenceTransformer(GTE_MODEL_ID, revision=GTE_REVISION, trust_remote_code=True, device=DEVICE)
gte_model.max_seq_length = int(gte_tokenizer.model_max_length)
gte_embeddings = gte_model.encode(documents['document_text'].tolist(), batch_size=GTE_BATCH_SIZE,
                            normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
gte_embeddings = np.ascontiguousarray(gte_embeddings, dtype=np.float32)

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Batches:   0%|          | 0/159 [00:00<?, ?it/s]

### 5.2. Faiss indeksiranje

In [32]:
gte_dir = INDEX_DIR / 'gte'
gte_dir.mkdir(parents=True, exist_ok=True)
product_ids = documents['logical_product_id'].tolist()
mapping = pd.DataFrame({"faiss_row": np.arange(len(product_ids)), "logical_product_id": list(product_ids)})
mapping.to_csv(gte_dir / "product_rows.csv", index=False)

gte_index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)
gte_index.add(gte_embeddings)
faiss.write_index(gte_index, str(gte_dir / "products.faiss"))

### 5.3. Embedovanje upita, pretraga top 5 proizvoda po upitu

In [33]:
gte_query_embeddings = gte_model.encode(queries["query_text"].tolist(),batch_size=GTE_BATCH_SIZE,
                                        normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
gte_query_embeddings = np.ascontiguousarray(gte_query_embeddings,dtype=np.float32)
distances, idx = gte_index.search(gte_query_embeddings,DEPTH)

results = []
for pos, query in queries.reset_index(drop=True).iterrows():
    ranked = pd.DataFrame({
        "logical_product_id": (mapping.iloc[ idx[pos] ]["logical_product_id"].values),
        "score": distances[pos]})

    ranked = ranked.merge(products,on="logical_product_id",how="left",validate="one_to_one")
    ranked.insert(0,"rank",np.arange(1, len(ranked) + 1))
    ranked.insert(0, "query_text", query["query_text"])
    ranked.insert(0, "query_id", query["query_id"])
    ranked.insert(0, "candidate_id", "gte_full")
    results.append(ranked)

gte_rankings = pd.concat(results,ignore_index=True)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

## 6. Priprema E5 chunkova


Formira se zajedničko zaglavlje sa metapodacima proizvoda (naziv, brend, kategorija). 

Zatim se kao odvojeni izvori obrađuju:
- **description**
- **pros**, **cons**, **best_uses**

Ako zaglavlje i izvor zauzimaju manje od 512 tokena, čuvaju se kao jedan chunk.
Ako ne, računa se raspoloživ broj tokena za sadržaj oduzimanjem zaglavlja, a sadržaj se deli sa overlapom od 64 tokena.
Metapodaci se ponavljaju u svakom chunku kako bi zadržao i osnovni kontekst.

In [35]:
chunk_rows = []

for _, product in documents.iterrows():
    product_id = product["logical_product_id"]
    metadata = "\n".join([f"Name: {product['product_name']}",f"Brand: {product['brand']}",f"Category: {product['category']}"])
    sources = {
        "description": (f"Description: {product['description']}"if str(product["description"]).strip()else ""),
        "platform_summary": "\n".join([f"Pros: {product['pros']}",f"Cons: {product['cons']}",f"Best uses: {product['best_uses']}"])
            if any(str(product[column]).strip() for column in ["pros", "cons", "best_uses"]) else ""}

    for source, content in sources.items():
        if not content:
            continue
        full_text = f"{metadata}\n{content}"
        full_length = len(e5_tokenizer("passage: " + full_text)["input_ids"])

        if full_length <= E5_INPUT_LIMIT:
            chunk_texts = [full_text]

        else:
            fixed_text = f"{metadata}\n"
            fixed_length  = len(e5_tokenizer("passage: " + fixed_text)["input_ids"])
            budget = (E5_INPUT_LIMIT-fixed_length)

            encoded = e5_tokenizer(content,add_special_tokens=False,truncation=True,max_length=budget,
                                   stride=E5_OVERLAP_TOKENS,return_overflowing_tokens=True)
            chunk_texts = [fixed_text+ e5_tokenizer.decode(
                                    token_ids,clean_up_tokenization_spaces=False).strip()
                                    for token_ids in encoded["input_ids"]]

        for position, chunk_text in enumerate(chunk_texts):
            token_count = len(e5_tokenizer("passage: " + chunk_text)["input_ids"])
            chunk_rows.append({
                "logical_product_id": product_id,
                "chunk_source": source,
                "token_count": token_count,
                "chunk_text": chunk_text
            })

chunks = pd.DataFrame(chunk_rows)
chunks.insert(0,"faiss_row",np.arange(len(chunks)))
assert chunks["token_count"].max() <= E5_INPUT_LIMIT

print("Number of chunks:", len(chunks))
display(chunks["token_count"].describe())
display(chunks.groupby("chunk_source").size().rename("chunk_count"))

Number of chunks: 2595


,token_count
count,2595.000000
mean,207.788439
std,111.827127
min,32.000000
25%,126.500000
50%,173.000000
75%,270.000000
max,512.000000


,chunk_count
chunk_source,
description,1335
platform_summary,1260


### Rezultat

Dobijeno je 2595 E5 chunkova:

- 1335 iz opisa
- 1260 iz platformskih sažetaka

Prosečan chunk ima približno 207 tokena.

## 7. E5-MaxP i E5-MeanChunks

### 7.1. E5 embedding chunkova

In [ ]:
e5_model = SentenceTransformer(E5_MODEL_ID,revision=E5_REVISION,device=DEVICE)
chunk_embeddings = e5_model.encode(("passage: " + chunks["chunk_text"]).tolist(),batch_size=E5_BATCH_SIZE,normalize_embeddings=True,
                                   convert_to_numpy=True,show_progress_bar=True)
chunk_embeddings = np.ascontiguousarray(chunk_embeddings,dtype=np.float32)
assert chunk_embeddings.shape == (len(chunks),EMBEDDING_DIMENSION)


Batches:   0%|          | 0/163 [00:00<?, ?it/s]

### 7.2. E5-MaxP indeks

- Čuva jedan vektor po chunk-u 
- Upit se poredi sa svim chunkovima 
- Konačni skor proizvoda predstavlja najveću sličnost između upita i bilo kog njegovog chunka


In [ ]:
maxp_dir = INDEX_DIR / "e5_maxp"
maxp_dir.mkdir(parents=True, exist_ok=True)

maxp_index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)
maxp_index.add(chunk_embeddings)
faiss.write_index(maxp_index,str(maxp_dir / "chunks.faiss"))

chunks.to_csv(maxp_dir / "chunk_rows.csv",index=False)


### 7.3. E5-MeanChunks indeks

- Chunkovi se grupišu prema logical_product_id. 
- Za svaki proizvod računa se prosek embeddinga njegovih chunkova, a prosečni vektor se zatim normalizuje.

Prosečni vektor daje uopšteniju predstavu proizvoda, ali može ublažiti specifičnu osobinu.

In [ ]:
product_chunks= chunks.groupby("logical_product_id", sort=False).indices
mean_embeddings = np.array([chunk_embeddings[product_chunks[product_id]].mean(axis=0) for product_id in product_ids] ,dtype=np.float32)
mean_embeddings = np.ascontiguousarray(mean_embeddings,dtype=np.float32)
faiss.normalize_L2(mean_embeddings)
assert mean_embeddings.shape == (EXPECTED_PRODUCTS,EMBEDDING_DIMENSION)

mean_dir = INDEX_DIR / "e5_meanchunks"
mean_dir.mkdir(parents=True, exist_ok=True)
mean_mapping = pd.DataFrame({"faiss_row": np.arange(len(product_ids)),"logical_product_id": product_ids})
mean_mapping.to_csv(mean_dir / "product_rows.csv",index=False)

mean_index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)
mean_index.add(mean_embeddings)
faiss.write_index(mean_index,str(mean_dir / "products.faiss"),)


### 7.4. E5 pretraga i svođenje rezultata na proizvode

In [ ]:
e5_query_inputs = ("query: " + queries["query_text"]).tolist()
e5_query_embeddings = e5_model.encode(e5_query_inputs,batch_size=E5_BATCH_SIZE,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
e5_query_embeddings = np.ascontiguousarray(e5_query_embeddings,dtype=np.float32)

maxp_scores, maxp_indices = maxp_index.search(e5_query_embeddings,maxp_index.ntotal)
mean_scores, mean_indices = mean_index.search(e5_query_embeddings,DEPTH)

maxp_results = []
mean_results = []

for pos, query in queries.reset_index(drop=True).iterrows():
    maxp_ranked = pd.DataFrame({
        "logical_product_id": (chunks.iloc[maxp_indices[pos]]["logical_product_id"].values),
        "score": maxp_scores[pos]})
    
    maxp_ranked = maxp_ranked.groupby("logical_product_id",as_index=False)["score"].max().sort_values("score",ascending=False).head(DEPTH).reset_index(drop=True)
    maxp_ranked = maxp_ranked.merge(products,on="logical_product_id",how="left",validate="one_to_one")
    maxp_ranked.insert(0,"rank",np.arange(1, len(maxp_ranked) + 1))
    maxp_ranked.insert(0,"query_text",query["query_text"])
    maxp_ranked.insert(0,"query_id",query["query_id"])
    maxp_ranked.insert(0,"candidate_id","e5_maxp")
    maxp_results.append(maxp_ranked)


    mean_ranked = pd.DataFrame({
        "logical_product_id": (mean_mapping.iloc[mean_indices[pos]]["logical_product_id"].values),
        "score": mean_scores[pos]})
    mean_ranked = mean_ranked.merge(products,on="logical_product_id",how="left",validate="one_to_one",)
    mean_ranked.insert(0,"rank",np.arange(1, len(mean_ranked) + 1))
    mean_ranked.insert(0,"query_text",query["query_text"])
    mean_ranked.insert(0,"query_id",query["query_id"])
    mean_ranked.insert(0,"candidate_id","e5_meanchunks")
    mean_results.append(mean_ranked)

maxp_rankings = pd.concat(maxp_results,ignore_index=True)
mean_rankings = pd.concat(mean_results,ignore_index=True)


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

## 8. Čuvanje rezultata 

Rang liste tri sistema u all_rankings.csv

In [ ]:
all_rankings = pd.concat([gte_rankings,maxp_rankings,mean_rankings],ignore_index=True)
all_rankings.to_csv(EVAL_DIR / "all_rankings.csv",index=False)

## 9. Formiranje evaluacionog bazena

In [42]:
judge_pool = all_rankings[["query_id", "logical_product_id"]].drop_duplicates().merge(
    queries[["query_id", "query_text", "query_type"]],on="query_id",validate="many_to_one").merge(
    documents[["logical_product_id","product_name","brand","category","description","pros","cons","best_uses"]],on="logical_product_id",validate="many_to_one")

judge_pool = judge_pool.sort_values(["query_id", "logical_product_id"]).sample(frac=1, random_state=SEED).reset_index(drop=True)
llm_test_pool = judge_pool.groupby("query_type").apply(lambda group: group.sample(n=5, random_state=SEED)).reset_index(drop=True)

judge_pool.to_csv(EVAL_DIR / 'judge_pool.csv', index=False)
llm_test_pool.to_csv(EVAL_DIR / 'llm_test_pool.csv', index=False)

/tmp/ipykernel_2563/2245188546.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  llm_test_pool = judge_pool.groupby("query_type").apply(lambda group: group.sample(n=5, random_state=SEED)).reset_index(drop=True)
